# 04 - Demo end-to-end

Flujo completo en AWS:
1. Subir imagen del test set a `s3://mlemons-raw/inference/<lote>/img.jpg`.
2. Disparar Step Function con `{lote_id, images: [...]}`.
3. SFN invoca Lambda `invoke_modelo_a` -> endpoint SageMaker -> predicciones.
4. SFN invoca Lambda `compute_score` -> score + DynamoDB write.
5. Leer registro en DynamoDB y validar.

Pre-requisitos: notebook 03 ejecutado y Model Package aprobado, `terraform apply` con `-var model_package_arn=<arn>`.

## Sobre el lote en esta demo

En produccion un lote seria una foto fisica de varios limones acomodados en banda. El dataset sintetico para componer lotes digitales se recorto del proyecto (decision D5 revisada el 2026-05-12). Para esta demo se simula un lote tomando N imagenes individuales del Lemon Quality Dataset de Kaggle y tratandolas como si fueran las detecciones de un lote real. El pipeline corre identico: cada imagen pasa por el endpoint, se acumulan las predicciones y se calcula un score promedio.

In [1]:
import json
import os
import subprocess
import time
import uuid
import boto3

creds = subprocess.check_output(
    ['aws', 'configure', 'export-credentials', '--format', 'env-no-export']
).decode()
for line in creds.strip().split('\n'):
    k, _, v = line.partition('=')
    os.environ[k] = v

region = 'us-east-2'
sess = boto3.Session(region_name=region)
account = sess.client('sts').get_caller_identity()['Account']

s3 = sess.client('s3')
sfn = sess.client('stepfunctions')
dynamodb = sess.resource('dynamodb')

BUCKET_RAW = f'mlemons-raw-{account}'
STATE_MACHINE_ARN = f'arn:aws:states:{region}:{account}:stateMachine:mlemons-pipeline'
TABLE = dynamodb.Table('mlemons_lotes')
print('Bucket   :', BUCKET_RAW)
print('SFN ARN  :', STATE_MACHINE_ARN)

Bucket   : mlemons-raw-898769391977
SFN ARN  : arn:aws:states:us-east-2:898769391977:stateMachine:mlemons-pipeline


## Seleccionar imagenes del test set

Tomo 4 buenas y 2 malas del dataset local para simular un lote mixto.

In [2]:
import glob
import random
random.seed(42)

good = sorted(glob.glob('../data/lemons/good_quality/*.jpg'))
bad  = sorted(glob.glob('../data/lemons/bad_quality/*.jpg'))
muestras = random.sample(good, 4) + random.sample(bad, 2)
print('Imagenes seleccionadas:', len(muestras))
for p in muestras:
    print(' ', p)

Imagenes seleccionadas: 6
  ../data/lemons/good_quality/good_quality_191.jpg
  ../data/lemons/good_quality/good_quality_1043.jpg
  ../data/lemons/good_quality/good_quality_493.jpg
  ../data/lemons/good_quality/good_quality_437.jpg
  ../data/lemons/bad_quality/bad_quality_303.jpg
  ../data/lemons/bad_quality/bad_quality_226.jpg


## Subir imagenes a S3 y armar evento

In [3]:
lote_id = f'demo-{uuid.uuid4().hex[:8]}'
images_s3 = []
for path in muestras:
    name = os.path.basename(path)
    key = f'inference/{lote_id}/{name}'
    s3.upload_file(path, BUCKET_RAW, key)
    images_s3.append(f's3://{BUCKET_RAW}/{key}')
print('Lote id:', lote_id)
print('Imagenes:')
for u in images_s3:
    print(' ', u)

Lote id: demo-c8ae1e17
Imagenes:
  s3://mlemons-raw-898769391977/inference/demo-c8ae1e17/good_quality_191.jpg
  s3://mlemons-raw-898769391977/inference/demo-c8ae1e17/good_quality_1043.jpg
  s3://mlemons-raw-898769391977/inference/demo-c8ae1e17/good_quality_493.jpg
  s3://mlemons-raw-898769391977/inference/demo-c8ae1e17/good_quality_437.jpg
  s3://mlemons-raw-898769391977/inference/demo-c8ae1e17/bad_quality_303.jpg
  s3://mlemons-raw-898769391977/inference/demo-c8ae1e17/bad_quality_226.jpg


## Disparar Step Function

In [4]:
payload = {'lote_id': lote_id, 'images': images_s3, 'threshold': 75}
resp = sfn.start_execution(
    stateMachineArn=STATE_MACHINE_ARN,
    name=lote_id,
    input=json.dumps(payload),
)
execution_arn = resp['executionArn']
print('Execution ARN:', execution_arn)

Execution ARN: arn:aws:states:us-east-2:898769391977:execution:mlemons-pipeline:demo-c8ae1e17


## Esperar a que termine

In [5]:
for _ in range(60):
    desc = sfn.describe_execution(executionArn=execution_arn)
    if desc['status'] != 'RUNNING':
        break
    time.sleep(2)
print('Status:', desc['status'])
print('Output:', desc.get('output'))

Status: SUCCEEDED
Output: {"statusCode":200,"body":"{\"lote_id\": \"demo-c8ae1e17\", \"score\": 66.67, \"decision\": \"RECHAZADO\", \"distribucion\": {\"good_quality\": 4, \"bad_quality\": 2}, \"n_images\": 6}"}


## Tiempos del pipeline

Cada step de la Step Function (`InvokeModeloA` y `ComputeScore`) tiene su propio evento de entrada y salida. Calculamos cuanto tomo cada uno y la latencia total del lote.

In [6]:
history = sfn.get_execution_history(executionArn=execution_arn)['events']
starts = {}
durations = {}
for ev in history:
    t = ev['timestamp']
    if ev['type'] == 'TaskStateEntered':
        name = ev['stateEnteredEventDetails']['name']
        starts[name] = t
    elif ev['type'] == 'TaskStateExited':
        name = ev['stateExitedEventDetails']['name']
        if name in starts:
            durations[name] = (t - starts[name]).total_seconds()
total = (desc['stopDate'] - desc['startDate']).total_seconds()
for name, secs in durations.items():
    print(f'  {name:14s}: {secs:.2f}s')
print(f'  {"TOTAL":14s}: {total:.2f}s')

  InvokeModeloA : 1.54s
  ComputeScore  : 0.88s
  TOTAL         : 2.48s


## Validar registro en DynamoDB

In [7]:
item = TABLE.get_item(Key={'lote_id': lote_id}).get('Item')
assert item is not None, 'No se persistio el lote en DynamoDB'
for k, v in item.items():
    print(f'  {k:14s} = {v}')

  decision       = RECHAZADO
  threshold      = 75.0
  distribucion   = {'bad_quality': Decimal('2'), 'good_quality': Decimal('4')}
  timestamp      = 1778637723
  score          = 66.67
  n_images       = 6
  lote_id        = demo-c8ae1e17


## Score esperado vs obtenido

Como las imagenes vienen del dataset etiquetado podemos calcular el score esperado (suponiendo que el modelo no se equivoca) y compararlo con el que entrego el pipeline. La diferencia entre los dos es indicativa del error del modelo en este lote.

In [8]:
import sys
sys.path.insert(0, '../src/lambda_code/compute_score')
from scoring_logic import score_lote  # mismo import que usa la Lambda en runtime

predicciones_esperadas = []
for path in muestras:
    clase = 'good_quality' if 'good_quality' in path else 'bad_quality'
    predicciones_esperadas.append({'predicted_class': clase})
esperado = score_lote(predicciones_esperadas, threshold=75)
obtenido = {
    'score': float(item['score']),
    'decision': item['decision'],
    'distribucion': {k: int(v) for k, v in item['distribucion'].items()},
    'n_images': int(item['n_images']),
}
print('Esperado (segun etiquetas del dataset):')
print(f'  score = {esperado["score"]} -> {esperado["decision"]}')
print(f'  distribucion = {esperado["distribucion"]}')
print()
print('Obtenido (segun predicciones del modelo):')
print(f'  score = {obtenido["score"]} -> {obtenido["decision"]}')
print(f'  distribucion = {obtenido["distribucion"]}')
print()
delta = obtenido['score'] - esperado['score']
print(f'Delta score: {delta:+.2f}')
print('Coincide decision:', obtenido['decision'] == esperado['decision'])

Esperado (segun etiquetas del dataset):
  score = 66.67 -> RECHAZADO
  distribucion = {'good_quality': 4, 'bad_quality': 2}

Obtenido (segun predicciones del modelo):
  score = 66.67 -> RECHAZADO
  distribucion = {'bad_quality': 2, 'good_quality': 4}

Delta score: +0.00
Coincide decision: True


## Cleanup opcional

Si vas a re-correr el demo, borrar la entrada del lote y las imagenes en S3.

In [9]:
# TABLE.delete_item(Key={'lote_id': lote_id})
# for u in images_s3:
#     key = u.replace(f's3://{BUCKET_RAW}/', '')
#     s3.delete_object(Bucket=BUCKET_RAW, Key=key)
# print('cleanup ok')

## Resumen

Lo que demuestra esta corrida:

- El modelo entrenado en SageMaker esta servido en un endpoint managed (`mlemons-modelo-a-endpoint`, `ml.t2.medium`).
- Las dos Lambdas (`invoke_modelo_a` y `compute_score`) corren dentro de una Step Function (`mlemons-pipeline`)   que orquesta inferencia + scoring + persistencia.
- El score y la decision del lote quedan en DynamoDB (`mlemons_lotes`) con su timestamp.
- Toda la infraestructura es Terraform; el endpoint se levanta y se baja con `make endpoint-up` / `make endpoint-down`.

Costos relevantes: el endpoint `ml.t2.medium` cuesta ~$0.06/hora encendido. Apagarlo entre demos con `make endpoint-down` evita gastos pasivos.